# 04 - Classification Models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, accuracy_score,
                             classification_report)

SEED = 42
df = pd.read_csv("../Data/creditcard_clean.csv")
X = df.drop(columns=["Class", "Hour"])
y = df["Class"]
print(X.shape, y.value_counts().to_dict())

## 1. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", X_train.shape, "fraud:", y_train.sum())
print("Test :", X_test.shape, "fraud:", y_test.sum())

## 2. Scaling

In [ ]:
scaler = RobustScaler().fit(X_train[["Time", "Amount"]])
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[["Time", "Amount"]] = scaler.transform(X_train[["Time", "Amount"]])
X_test_s[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

## 3. Undersampling (training set only)

In [ ]:
rng = np.random.RandomState(SEED)
fraud_idx = y_train.index[y_train == 1]
legit_idx = y_train.index[y_train == 0]
legit_keep = rng.choice(legit_idx, size=10 * len(fraud_idx), replace=False)
keep = np.concatenate([fraud_idx, legit_keep])
rng.shuffle(keep)

X_res, y_res = X_train_s.loc[keep], y_train.loc[keep]
print("Resampled train:", X_res.shape, y_res.value_counts().to_dict())

## 4. Evaluation Helper

In [ ]:
def get_scores(model, X_):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_)[:, 1]
    return model.decision_function(X_)

def evaluate(name, model):
    pred = model.predict(X_test_s)
    score = get_scores(model, X_test_s)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, score),
        "PR-AUC": average_precision_score(y_test, score),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }, pred, score

all_results, store = [], {}

## 5. Train 5 Classifiers (default)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "SVM": SVC(kernel="rbf", random_state=SEED),
}

base_rows = []
for name, m in models.items():
    m.fit(X_res, y_res)
    r, _, _ = evaluate(f"{name} (default)", m)
    base_rows.append(r)
all_results.extend(base_rows)

pd.DataFrame(base_rows).set_index("Model").round(4)

## 6. Hyperparameter Tuning (GridSearchCV)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

param_grids = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, random_state=SEED),
                            {"C": [0.01, 0.1, 1, 10]}),
    "KNN": (KNeighborsClassifier(),
            {"n_neighbors": [3, 5, 7, 11], "weights": ["uniform", "distance"]}),
    "Naive Bayes": (GaussianNB(),
                    {"var_smoothing": [1e-11, 1e-9, 1e-7]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=SEED),
                      {"max_depth": [3, 5, 8, None], "min_samples_leaf": [1, 5, 10]}),
    "SVM": (SVC(random_state=SEED),
            {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"]}),
}

tuned, best_params = {}, {}
for name, (est, grid) in param_grids.items():
    gs = GridSearchCV(est, grid, scoring="average_precision", cv=cv, n_jobs=-1)
    gs.fit(X_res, y_res)
    tuned[name] = gs.best_estimator_
    best_params[name] = gs.best_params_
    print(f"{name:20s} best: {gs.best_params_} | CV AP = {gs.best_score_:.4f}")

## 7. Tuned Models on Test Set

In [ ]:
tuned_rows = []
for name, m in tuned.items():
    r, pred, score = evaluate(f"{name} (tuned)", m)
    tuned_rows.append(r)
    store[name] = {"pred": pred, "score": score}
all_results.extend(tuned_rows)

pd.DataFrame(tuned_rows).set_index("Model").round(4)

## 8. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for a, (name, d) in zip(axes, store.items()):
    cm = confusion_matrix(y_test, d["pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
    a.set_title(name)
plt.tight_layout(); plt.show()

## 9. Decision Tree Visualization

In [ ]:
dt_best = tuned["Decision Tree"]
print("Depth:", dt_best.get_depth(), "| Leaves:", dt_best.get_n_leaves())
print(export_text(dt_best, feature_names=list(X_res.columns), max_depth=4))

In [ ]:
plt.figure(figsize=(20, 9))
plot_tree(dt_best, feature_names=list(X_res.columns),
          class_names=["Legit", "Fraud"],
          filled=True, rounded=True, max_depth=3, fontsize=9)
plt.title("Decision Tree (top 3 levels)")
plt.show()

## 10. Save for Notebook 05

In [ ]:
results_df = pd.DataFrame(all_results).set_index("Model")
joblib.dump({"results": results_df, "store": store, "y_test": y_test.values,
             "best_params": best_params}, "../Data/results.pkl")
results_df.round(4)